# Verifikasi 95% Confidence Interval — Akurasi Klasifikasi AI

Menghitung Wilson dan Clopper-Pearson 95% CI untuk proporsi kesepakatan AI vs konsensus manusia (374 dari 407 akun konsensus).

In [1]:
# Install statsmodels kalau belum ada
!pip install statsmodels -q

In [2]:
from statsmodels.stats.proportion import proportion_confint

# Titik estimasi dari hasil validasi Anda
matched = 374   # AI cocok dengan konsensus manusia
total = 407     # akun konsensus (dua coder sepakat)

point = matched / total * 100
print(f'Point estimate: {point:.1f}%')

# Wilson (rekomendasi, default modern)
lo_w, hi_w = proportion_confint(count=matched, nobs=total, method='wilson')
print(f'Wilson 95% CI:          [{lo_w*100:.1f}, {hi_w*100:.1f}]')

# Clopper-Pearson (eksak, lebih konservatif)
lo_cp, hi_cp = proportion_confint(count=matched, nobs=total, method='beta')
print(f'Clopper-Pearson 95% CI: [{lo_cp*100:.1f}, {hi_cp*100:.1f}]')

Point estimate: 91.9%
Wilson 95% CI:          [88.8, 94.2]
Clopper-Pearson 95% CI: [88.8, 94.4]


In [3]:
# OPSIONAL: hitung ulang dari file validasi mentah untuk memastikan 374 dan 407 benar
import pandas as pd

def load(name):
    with open(name, encoding='utf-8-sig', errors='replace') as f:
        head = f.read(2048)
    sep = ';' if head.count(';') > head.count(',') else ','
    df = pd.read_csv(name, sep=sep, encoding='utf-8-sig')
    df.columns = [c.strip().lower() for c in df.columns]
    df['username'] = df['username'].astype(str).str.strip().str.lower()
    if 'coder_label' in df.columns:
        df['coder_label'] = df['coder_label'].astype(str).str.strip().str.lower()
    if 'account_type' in df.columns:
        df['account_type'] = df['account_type'].astype(str).str.strip().str.lower()
    return df

c1 = load('coding_coder1.csv')
c2 = load('coding_coder2.csv')
ai = load('ai_labels_HIDDEN.csv')

m = (c1[['username','coder_label']]
     .merge(c2[['username','coder_label']], on='username', suffixes=('_1','_2'))
     .merge(ai[['username','account_type','has_profile']], on='username'))

cons = m[m.coder_label_1 == m.coder_label_2]
n_matched = int((cons.account_type == cons.coder_label_1).sum())
n_total = len(cons)

print(f'Recomputed from files: matched = {n_matched}, total = {n_total}')

from statsmodels.stats.proportion import proportion_confint
lo, hi = proportion_confint(count=n_matched, nobs=n_total, method='wilson')
print(f'Wilson 95% CI: [{lo*100:.1f}, {hi*100:.1f}]')

Recomputed from files: matched = 374, total = 407
Wilson 95% CI: [88.8, 94.2]


## Yang dilaporkan di manuskrip

Angka yang muncul di §3.1 revisi:

- **Point estimate:** 91.9%
- **95% Wilson CI:** [88.8, 94.2]

Kalau sel keempat menghasilkan angka berbeda dari `matched=374, total=407`, gunakan angka baru itu dan hitung ulang CI-nya. Angka dari file mentah lebih otoritatif daripada angka yang ditulis manual.